# Phase 8 — Planning & Reasoning

> Run AFTER Phase 8 (restart kernel first). Cell 4 makes several Gemini calls (planner + agents + synthesis).

**Goal:** a compound query is DECOMPOSED into a plan BEFORE agents run, executed step-by-step, then synthesized with conflict-handling. Uses CLT-005 (holds individual stocks).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv('../.env')
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env'
print('env OK')

env OK


## 1. Complexity classifier (Strategy) — free heuristic

In [2]:
from app.graph.planner import HeuristicComplexityClassifier
clf = HeuristicComplexityClassifier()
for q in ['What stocks do I own?',
          'Should I rebalance given the Fed announcement and my risk tolerance?']:
    print(f'{clf.is_complex(q)!s:5}  {q}')

False  What stocks do I own?
True   Should I rebalance given the Fed announcement and my risk tolerance?


## 2. The planner decomposes a complex query into an ordered plan

In [3]:
from app.graph.planner import PlannerNode
from app.graph.router import AgentSpec
from langchain_core.messages import HumanMessage
specs = [AgentSpec('portfolio','holdings/allocations'), AgentSpec('market_research','news/sectors'),
         AgentSpec('securities_analysis','technical'), AgentSpec('risk','vol/beta/VaR/tolerance')]
out = PlannerNode(specs).run({'messages':[HumanMessage(content=
    'Should I rebalance given the recent Fed announcement and my risk tolerance?')]})
for i, s in enumerate(out['plan'], 1):
    print(f"{i}. {s['agent']:18} — {s['goal']}")

AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"steps": ["market_research:Analyze the recent Fed announcement and ", "portfolio:Retrieve current asset holdings and targ", "risk:Assess the user's risk tolerance and cur", "portfolio:Determine if rebalancing is necessary to"], "event": "planner_plan", "agent": "planner", "level": "info", "timestamp": "2026-07-13T01:34:23.223918Z"}
1. market_research    — Analyze the recent Fed announcement and its impact on current market sectors.
2. portfolio          — Retrieve current asset holdings and target allocation percentages.
3. risk               — Assess the user's risk tolerance and current portfolio volatility/beta.
4. portfolio          — Determine if rebalancing is necessary to align with risk tolerance and market conditions.


## 3. Full run through the graph — watch the plan execute then synthesize

In [4]:
from app.graph.builder import GraphBuilder
from app.agents.base import _text
graph = GraphBuilder().with_all().build()
cfg = {'configurable': {'thread_id': 'CLT-005-nb8'}}
final = None
for chunk in graph.stream({'messages':[HumanMessage(content=
      'Should I rebalance given the recent Fed announcement and my risk tolerance?')],
      'client_id':'CLT-005','session_id':'nb8'}, cfg, stream_mode='updates'):
    for node, upd in chunk.items():
        if node=='planner' and (upd or {}).get('plan'):
            print('PLAN:', [s['agent'] for s in upd['plan']])
        elif node in ('portfolio','market_research','securities_analysis','risk'):
            print(' ran', node)
        elif node=='synthesizer':
            final = (upd or {}).get('final_answer')
print('\n=== SYNTHESIZED ANSWER ===\n', final)

{"client_id": "CLT-005", "prior_decisions": 1, "event": "memory_read", "agent": "planner", "level": "info", "timestamp": "2026-07-13T01:34:53.893050Z"}
{"guard": "pii", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "input_guard", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:53.894598Z"}
{"guard": "prompt_injection", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "input_guard", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:53.894977Z"}
{"guard": "scope", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "input_guard", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:53.895232Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"steps": ["market_research:Analyze the recent Fed announcement and ", "portfolio:Retrieve the client's current holdings, ", "risk:Assess the client's current portfolio ri"], "event": "planner_plan", "client_id": "CLT-005", "agent": "planner", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:54.985895Z"}
PLAN: ['market_research', 'portfolio', 'risk']
{"step": 1, "of": 3, "agent": "market_research", "goal": "Analyze the recent Fed announcement and its impact on curren", "event": "supervisor_plan_step", "client_id": "CLT-005", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:54.987660Z"}
{"query": "Should I rebalance given the recent Fed announcement and my risk tolerance?", "event": "agent_start", "client_id": "CLT-005", "agent": "market_research", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:34:54.990566Z"}


HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


{"source": "finnhub", "ticker": "FED", "event": "news_source_empty", "client_id": "CLT-005", "agent": "market_research", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:35:03.345717Z"}
{"source": "yfinance", "ticker": "FED", "event": "news_source_empty", "client_id": "CLT-005", "agent": "market_research", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:35:03.563039Z"}


...

{"chars": 3893, "revision": 0, "had_critique": false, "event": "synthesized", "client_id": "CLT-005", "agent": "synthesizer", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:13.922233Z"}
{"guard": "numeric_consistency", "action": "revise", "passed": false, "reason": "answer contains numbers not found in tool evidence: [500.0]", "event": "guardrail_check", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:13.925429Z"}
{"attempt": 1, "guard": "numeric_consistency", "reason": "answer contains numbers not found in tool evidence: [500.0]", "event": "reflection_revise", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:13.927354Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


{"chars": 3611, "revision": 1, "had_critique": true, "event": "synthesized", "client_id": "CLT-005", "agent": "synthesizer", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:44.718122Z"}
{"guard": "numeric_consistency", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:44.721883Z"}
{"guard": "conflict_disclosure", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:36:44.724636Z"}


AFC is enabled with max remote calls: 10.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 503 Service Unavailable"
Retrying google.genai._api_client.BaseApiClient._request_once in 1.21 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.
HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


{"guard": "citation_coverage", "action": "pass", "passed": true, "reason": "", "event": "guardrail_check", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:37:40.701525Z"}
{"guard": "groundedness", "action": "pass", "passed": true, "reason": "no retrieved context", "event": "guardrail_check", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:37:40.702732Z"}
{"revisions": 1, "event": "reflection_pass", "client_id": "CLT-005", "agent": "reflector", "session_id": "nb8", "level": "info", "timestamp": "2026-07-13T01:37:40.703503Z"}
{"client_id": "CLT-005", "session_id": "nb8", "event": "decision_saved", "agent": "planner", "level": "info", "timestamp": "2026-07-13T01:37:40.719355Z"}

=== SYNTHESIZED ANSWER ===
 ### 1. Client Question
The client is asking whether they should rebalance their portfolio in light of the recent Federal Reserve announcement and their persona

## ✅ Acceptance check

In [5]:
assert out['plan'] and len(out['plan']) >= 2, 'complex query should yield a multi-step plan'
assert final, 'synthesizer should produce a final answer'
print('Phase 8 acceptance: PASS — plan printed before execution, then synthesized')

Phase 8 acceptance: PASS — plan printed before execution, then synthesized
